# 🌍 Hydrogeological Potential Mapping — Preprocessing & Scoring

> **Author :** HAMIDOU BÂ Abdoul Aziz. | Geologist · Product Owner · Data Scientist  
> **Contact :** [LinkedIn](https://www.linkedin.com/in/azioba) · [GitHub](https://github.com/azioba)  
> **License :** MIT — Free to use, adapt and share with attribution  
> **Version :** 2.0 | February 2026

---

## What does this notebook do?

This notebook implements a **Weighted Index Overlay (WIO)** model to map
groundwater potential zones at national scale using open-source data only.

It was originally developed for **Niger (West Africa)** and is designed
to be fully reusable for any country by changing the configuration section.

### Pipeline
```
Raw data (BGS geology + CHIRPS rainfall + SRTM DEM)
        ↓
Reprojection → common CRS & resolution
        ↓
Scoring (geology) + Normalization (0–1) per layer
        ↓
Weighted composite score (GWPI)
        ↓
4-class classification → Groundwater Potential Map
```

### Required input data (all free & open source)
| Layer | Source | Format |
|-------|--------|--------|
| Administrative boundaries | [GADM](https://gadm.org) | `.gpkg` |
| Hydrogeology | [BGS Africa Groundwater Atlas](https://www.bgs.ac.uk/africagroundwateratlas/) | `.shp` |
| Rainfall | [CHIRPS](https://www.chc.ucsb.edu/data/chirps) | `.tif` |
| Digital Elevation Model | [HydroSHEDS](https://www.hydrosheds.org) | `.tif` |

### Methodology reference
Weighted Index Overlay as described in:
- Andualem & Demeke (2020) — *Groundwater potential mapping using geospatial techniques* — Cogent Geoscience
- Hussein et al. (2021) — *Groundwater Potential Zone Mapping Using AHP and GIS* — PMC


---
## ⚙️ CONFIGURATION — Edit this section for your country


In [ ]:
# ════════════════════════════════════════════════════════════
#   CONFIGURATION — Adapt to your study area
# ════════════════════════════════════════════════════════════

from pathlib import Path

# ── Project paths ─────────────────────────────────────────
# Adapt ROOT to your local project folder
ROOT      = Path(r'C:/your/project/folder')   # ← CHANGE THIS
DATA_RAW  = ROOT / 'data' / 'raw'
DATA_PROC = ROOT / 'data' / 'processed'
MAPS      = ROOT / 'outputs' / 'maps'

for folder in [DATA_RAW, DATA_PROC, MAPS,
               DATA_RAW/'gadm', DATA_RAW/'bgs',
               DATA_RAW/'chirps', DATA_RAW/'srtm']:
    folder.mkdir(parents=True, exist_ok=True)

# ── Coordinate Reference System ────────────────────────────
# Use a projected CRS appropriate for your country (meters)
# Examples:
#   Niger / West Africa  → EPSG:32632 (UTM Zone 32N)
#   Kenya / East Africa  → EPSG:32637 (UTM Zone 37N)
#   Morocco / N. Africa  → EPSG:32629 (UTM Zone 29N)
#   Senegal              → EPSG:32628 (UTM Zone 28N)
#   Ethiopia             → EPSG:32637 (UTM Zone 37N)
# Find your UTM zone: https://epsg.io
TARGET_CRS = 'EPSG:32632'   # ← CHANGE THIS

# ── Spatial resolution ─────────────────────────────────────
# 1000m = 1km  → good for national-scale analysis
# 500m         → better for regional analysis (larger files)
# 250m         → use for watershed-scale analysis
RES = 1000   # meters — CHANGE IF NEEDED

# ── Input file paths ───────────────────────────────────────
# Adapt filenames to match your downloaded files
GADM_PATH  = DATA_RAW / 'gadm'   / 'gadm41_XXX.gpkg'     # ← CHANGE XXX to your country code
BGS_PATH   = DATA_RAW / 'bgs'    / 'YourCountry_HG.shp'  # ← CHANGE to your BGS shapefile
CHIRPS_DIR = DATA_RAW / 'chirps'                          # folder with chirps-v2.0.YYYY.tif files
DEM_PATH   = DATA_RAW / 'srtm'   / 'your_dem.tif'        # ← CHANGE to your DEM file

# ── BGS hydrogeology scoring ───────────────────────────────
# Map your BGS 'NigHGComb' (or equivalent) codes to scores 0-4.
# Check your shapefile attribute table to get the exact code values.
# General logic:
#   4 = Highest productivity (consolidated + fractured sedimentary)
#   3 = High productivity (unconsolidated sedimentary)
#   2 = Medium productivity (intergranular)
#   1 = Low productivity (basement/crystalline)
#   0 = Not assessed
#
# Niger example (NigHGComb field):
# HYDRO_SCORES = {
#     'CSIF-M/H': 4,
#     'CSI-M/H' : 3,
#     'U-M/H(L)': 3,
#     'I-L/M'   : 2,
#     'B-L'     : 1,
#     'n/a'     : 0,
# }
HYDRO_SCORES = {
    'CODE_HIGH'  : 4,   # ← CHANGE: replace with your BGS codes
    'CODE_MED_H' : 3,
    'CODE_MED'   : 2,
    'CODE_LOW'   : 1,
    'n/a'        : 0,
}
BGS_CODE_COLUMN = 'NigHGComb'  # ← CHANGE: column name in your BGS shapefile

# ── Model weights ──────────────────────────────────────────
# Must sum to 1.0
# Adjust based on your hydrogeological context:
#   - Arid regions  → increase rainfall weight
#   - Basement areas → increase lineament/slope weight
#   - Data-rich areas → calibrate with borehole data
WEIGHTS = {
    'geology' : 0.40,   # Primary control on aquifer presence
    'rainfall': 0.25,   # Recharge proxy
    'slope'   : 0.20,   # Infiltration vs runoff
    'tpi'     : 0.15,   # Topographic accumulation zones
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001, 'Weights must sum to 1.0'

# ── GADM layer names ───────────────────────────────────────
# Standard GADM layer names — usually no change needed
GADM_COUNTRY = 'ADM_ADM_0'   # country boundary
GADM_REGION  = 'ADM_ADM_1'   # first administrative level

print('✅ Configuration loaded')
print(f'   CRS    : {TARGET_CRS}')
print(f'   Res    : {RES}m')
print(f'   Weights: { {k: f"{v*100:.0f}%" for k, v in WEIGHTS.items()} }')

---
## 1. Imports & setup


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from scipy.ndimage import uniform_filter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

# ── Check dependencies ─────────────────────────────────────
missing = []
for pkg in ['numpy','geopandas','rasterio','scipy','matplotlib']:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'⚠️  Missing packages: {missing}')
    print(f'   Install with: pip install {" ".join(missing)}')
else:
    print('✅ All dependencies OK')

---
## 2. Load administrative boundaries


In [ ]:
# ── Load country & region boundaries ──────────────────────
country = gpd.read_file(GADM_PATH, layer=GADM_COUNTRY).to_crs(TARGET_CRS)
regions = gpd.read_file(GADM_PATH, layer=GADM_REGION ).to_crs(TARGET_CRS)

# ── Compute grid parameters ────────────────────────────────
bounds    = country.total_bounds  # [minx, miny, maxx, maxy]
WIDTH     = int((bounds[2] - bounds[0]) / RES)
HEIGHT    = int((bounds[3] - bounds[1]) / RES)
TRANSFORM = from_bounds(bounds[0], bounds[1],
                        bounds[2], bounds[3], WIDTH, HEIGHT)

# ── Country mask ───────────────────────────────────────────
mask = rasterize(
    [(country.union_all(), 1)],
    out_shape=(HEIGHT, WIDTH), transform=TRANSFORM,
    fill=0, dtype='uint8'
).astype(bool)

# ── Base raster profile ────────────────────────────────────
PROFILE = {
    'driver': 'GTiff', 'dtype': 'float32', 'width': WIDTH,
    'height': HEIGHT, 'count': 1, 'crs': TARGET_CRS,
    'transform': TRANSFORM, 'nodata': -9999
}

print(f'✅ Boundaries loaded')
print(f'   Grid      : {WIDTH} x {HEIGHT} pixels at {RES}m')
print(f'   Area      : {(mask.sum() * (RES/1000)**2):,.0f} km²')
print(f'   Regions   : {len(regions)}')

---
## 3. Layer 1 — Geology (BGS hydrogeology)


In [ ]:
# ── Load & inspect BGS shapefile ──────────────────────────
bgs = gpd.read_file(BGS_PATH).to_crs(TARGET_CRS)

print(f'BGS shapefile loaded: {len(bgs)} polygons')
print(f'\nUnique codes in [{BGS_CODE_COLUMN}]:')
print(bgs[BGS_CODE_COLUMN].value_counts())
print('\n⚠️  Check these codes match your HYDRO_SCORES dictionary in CONFIG')

In [ ]:
# ── Apply scores ───────────────────────────────────────────
bgs['score_geo'] = bgs[BGS_CODE_COLUMN].map(HYDRO_SCORES).fillna(0)

# Check for unmapped codes
unmapped = bgs[bgs['score_geo'].isna()][BGS_CODE_COLUMN].unique()
if len(unmapped) > 0:
    print(f'⚠️  Unmapped codes (assigned score 0): {unmapped}')
    print('   → Add these to HYDRO_SCORES in the CONFIG section')

print('Score distribution:')
bgs['area_km2'] = bgs.geometry.area / 1e6
summary = bgs.groupby('score_geo').agg(
    n_polygons=('score_geo','count'),
    area_km2=('area_km2','sum')
).round(0)
print(summary)

# ── Rasterize ──────────────────────────────────────────────
shapes_geo = [(geom, score) for geom, score
              in zip(bgs.geometry, bgs['score_geo'])
              if geom is not None]

geo_raster = rasterize(
    shapes_geo, out_shape=(HEIGHT, WIDTH),
    transform=TRANSFORM, fill=0, dtype='float32'
)
geo_raster = np.where(mask, geo_raster, np.nan)

# Normalize 0-1
max_score  = max(HYDRO_SCORES.values())
geo_norm   = np.where(mask, geo_raster / max_score, np.nan)

with rasterio.open(DATA_PROC / 'layer_geology.tif', 'w', **PROFILE) as dst:
    dst.write(geo_norm.astype('float32'), 1)

print(f'\n✅ Geology rasterized & normalized')
print(f'   Saved: data/processed/layer_geology.tif')

---
## 4. Layer 2 — Rainfall (CHIRPS)


In [ ]:
# ── Load & reproject CHIRPS files ─────────────────────────
# Expects files named: chirps-v2.0.YYYY.tif
chirps_files = sorted(CHIRPS_DIR.glob('chirps-v2.0.*.tif'))
print(f'CHIRPS files found: {len(chirps_files)}')
for f in chirps_files:
    print(f'   {f.name}')

if len(chirps_files) == 0:
    raise FileNotFoundError(
        f'No CHIRPS files found in {CHIRPS_DIR}\n'
        f'Expected format: chirps-v2.0.YYYY.tif\n'
        f'Download from: https://www.chc.ucsb.edu/data/chirps'
    )

arrays = []
for f in chirps_files:
    with rasterio.open(f) as src:
        data = np.zeros((HEIGHT, WIDTH), dtype='float32')
        reproject(
            source=rasterio.band(src, 1), destination=data,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=TRANSFORM, dst_crs=TARGET_CRS,
            resampling=Resampling.bilinear
        )
        arr = data.astype(float)
        arr[arr < 0] = np.nan
        arrays.append(arr)

# Multi-year mean
chirps_mean = np.nanmean(np.stack(arrays, axis=0), axis=0)
chirps_mean = np.where(mask, chirps_mean, np.nan)

# Normalize 0-1 (higher rainfall = better recharge)
c_min, c_max = np.nanmin(chirps_mean), np.nanmax(chirps_mean)
chirps_norm  = (chirps_mean - c_min) / (c_max - c_min)
chirps_norm  = np.where(mask, chirps_norm, np.nan)

with rasterio.open(DATA_PROC / 'layer_rainfall.tif', 'w', **PROFILE) as dst:
    dst.write(chirps_norm.astype('float32'), 1)

print(f'\nRainfall range: {c_min:.0f} – {c_max:.0f} mm/yr')
print(f'✅ Rainfall normalized & saved')

---
## 5. Layers 3 & 4 — Topography (Slope + TPI)


In [ ]:
# ── Reproject DEM to target CRS ───────────────────────────
dem_utm = np.zeros((HEIGHT, WIDTH), dtype='float32')

with rasterio.open(DEM_PATH) as src:
    reproject(
        source=rasterio.band(src, 1), destination=dem_utm,
        src_transform=src.transform, src_crs=src.crs,
        dst_transform=TRANSFORM, dst_crs=TARGET_CRS,
        resampling=Resampling.bilinear
    )

dem_utm = dem_utm.astype(float)
dem_utm[dem_utm < -100] = np.nan  # remove nodata values
dem_utm = np.where(mask, dem_utm, np.nan)
dem_filled = np.where(np.isnan(dem_utm), 0, dem_utm)

print(f'DEM reprojected: {WIDTH}x{HEIGHT} pixels')
print(f'Elevation range: {np.nanmin(dem_utm):.0f} – {np.nanmax(dem_utm):.0f} m')

In [ ]:
# ── Slope ──────────────────────────────────────────────────
# Low slope = high infiltration = good for groundwater
# → inverted normalization
dy, dx = np.gradient(dem_filled, RES, RES)
slope   = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
slope   = np.where(np.isnan(dem_utm), np.nan, slope)

s_min, s_max = np.nanmin(slope), np.nanmax(slope)
slope_norm   = 1 - (slope - s_min) / (s_max - s_min)  # inverted
slope_norm   = np.where(mask, slope_norm, np.nan)

with rasterio.open(DATA_PROC / 'layer_slope.tif', 'w', **PROFILE) as dst:
    dst.write(slope_norm.astype('float32'), 1)

# ── TPI — Topographic Position Index ──────────────────────
# Negative TPI = low areas (valleys/depressions) = water accumulation
# → inverted normalization
# Neighborhood size: ~15 pixels (~15km at 1km resolution)
# Adjust 'size' parameter based on your resolution & landscape scale
neighborhood = uniform_filter(dem_filled, size=15)
tpi          = dem_utm - neighborhood

tpi_min, tpi_max = np.nanmin(tpi), np.nanmax(tpi)
tpi_norm = 1 - (tpi - tpi_min) / (tpi_max - tpi_min)  # inverted
tpi_norm = np.where(mask, tpi_norm, np.nan)

with rasterio.open(DATA_PROC / 'layer_tpi.tif', 'w', **PROFILE) as dst:
    dst.write(tpi_norm.astype('float32'), 1)

print(f'✅ Slope normalized (max: {s_max:.1f}°, mean: {np.nanmean(slope):.1f}°)')
print(f'✅ TPI normalized (range: {tpi_min:.0f}m – {tpi_max:.0f}m)')

---
## 6. Composite score & classification


In [ ]:
# ── Weighted composite score (GWPI) ───────────────────────
score = (
    WEIGHTS['geology']  * geo_norm    +
    WEIGHTS['rainfall'] * chirps_norm +
    WEIGHTS['slope']    * slope_norm  +
    WEIGHTS['tpi']      * tpi_norm
)
score = np.where(mask, score, np.nan)

# ── Quartile-based classification ─────────────────────────
# Quartiles ensure equal area in each class
# Alternative: use fixed thresholds if you have field validation data
q25, q50, q75 = [np.nanpercentile(score, p) for p in [25, 50, 75]]

classes = np.full_like(score, np.nan)
classes = np.where(score <= q25,                   1, classes)  # Low
classes = np.where((score > q25) & (score <= q50), 2, classes)  # Moderate
classes = np.where((score > q50) & (score <= q75), 3, classes)  # Good
classes = np.where(score > q75,                    4, classes)  # Excellent
classes = np.where(mask, classes, np.nan)

# ── Save ───────────────────────────────────────────────────
with rasterio.open(DATA_PROC / 'score_composite.tif', 'w', **PROFILE) as dst:
    dst.write(score.astype('float32'), 1)
with rasterio.open(DATA_PROC / 'potentiel_hydro_classes.tif', 'w', **PROFILE) as dst:
    dst.write(classes.astype('float32'), 1)

print(f'GWPI range: {np.nanmin(score):.3f} – {np.nanmax(score):.3f}')
print(f'Thresholds: Q25={q25:.3f} | Q50={q50:.3f} | Q75={q75:.3f}')
print()

LABELS = {1: 'Low', 2: 'Moderate', 3: 'Good', 4: 'Excellent'}
pixel_km2 = (RES / 1000) ** 2
print('Area by class:')
for cls, label in LABELS.items():
    n   = np.sum(classes == cls)
    a   = n * pixel_km2
    pct = n / mask.sum() * 100
    print(f'  {cls} — {label:<10} : {a:>10,.0f} km²  ({pct:.1f}%)')

---
## 7. Output map


In [ ]:
# ── Final groundwater potential map ───────────────────────
fig, ax = plt.subplots(figsize=(12, 13))

COLORS  = ['#d73027', '#fc8d59', '#91cf60', '#1a9850']
cmap_hg = mcolors.ListedColormap(COLORS)
norm_hg = mcolors.BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], cmap_hg.N)
ext_utm = [bounds[0], bounds[2], bounds[1], bounds[3]]

ax.imshow(np.ma.masked_invalid(classes), extent=ext_utm,
          cmap=cmap_hg, norm=norm_hg, origin='upper')

country.boundary.plot(ax=ax, color='black', linewidth=1.5, zorder=3)
regions.boundary.plot(ax=ax, color='#333', linewidth=0.5,
                       linestyle='--', zorder=3)

# Region labels — auto-detect name column
col_nom = next((c for c in regions.columns
                if 'name' in c.lower()), None)
if col_nom:
    for _, row in regions.iterrows():
        c = row.geometry.centroid
        ax.annotate(row[col_nom].upper(), xy=(c.x, c.y),
                    ha='center', fontsize=7, fontweight='bold',
                    color='white',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='#222', alpha=0.65,
                              edgecolor='none'), zorder=5)

# Legend
legend_patches = [
    mpatches.Patch(color=COLORS[i], label=f'{i+1} — {LABELS[i+1]}')
    for i in range(4)
]
ax.legend(handles=legend_patches, loc='lower left',
          fontsize=10, framealpha=0.9,
          title='Groundwater Potential', title_fontsize=11)

# Methodology note
note = (
    f'Model: Geology BGS ({WEIGHTS["geology"]*100:.0f}%) '
    f'+ Rainfall CHIRPS ({WEIGHTS["rainfall"]*100:.0f}%)\n'
    f'       + Slope SRTM ({WEIGHTS["slope"]*100:.0f}%) '
    f'+ TPI ({WEIGHTS["tpi"]*100:.0f}%)\n'
    f'Resolution: {RES//1000} km | CRS: {TARGET_CRS}'
)
ax.text(0.98, 0.02, note, transform=ax.transAxes,
        fontsize=8, va='bottom', ha='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

# Signature
ax.text(0.01, 0.01,
        'Sensei A. | github.com/azioba',
        transform=ax.transAxes, fontsize=7,
        color='#555', va='bottom', style='italic')

ax.set_title(
    'Groundwater Potential Zone Map\n'
    'Weighted Index Overlay Model — Open Source Data',
    fontsize=14, fontweight='bold'
)
ax.set_xlabel(f'X {TARGET_CRS} (m)', fontsize=10)
ax.set_ylabel(f'Y {TARGET_CRS} (m)', fontsize=10)
ax.ticklabel_format(style='sci', axis='both', scilimits=(0,0))

plt.tight_layout()
out_map = MAPS / 'groundwater_potential_map.png'
plt.savefig(out_map, dpi=200, bbox_inches='tight')
plt.show()
print(f'💾 {out_map}')

In [ ]:
# ── Final summary ─────────────────────────────────────────
print('=' * 60)
print('  SUMMARY')
print('=' * 60)
print('\n  Layers produced:')
for f in sorted(DATA_PROC.glob('layer_*.tif')):
    print(f'    ✅ {f.name}')
print('  Composite score:')
for f in sorted(DATA_PROC.glob('*composite*')):
    print(f'    ✅ {f.name}')
print('\n  Model weights:')
for k, v in WEIGHTS.items():
    print(f'    {k:<12}: {v*100:.0f}%')
print()
print('  Next steps:')
print('    → Run 03_interactive_map.ipynb  (Folium web map)')
print('    → Add lineament density layer   (GEM faults + DEM gradients)')
print('    → Add NDVI layer                (MODIS or Sentinel-2)')
print('    → Validate against borehole data')
print()
print('  Author : HAMIDOU BÂ Abdoul Aziz.')
print('  GitHub : github.com/azioba')
print('=' * 60)